Connected to torch_env (Python 3.10.14)

In [1]:
"""
Created on 12 Dec 2023

Obtaining the Physics Residuals as a measure of UQ on DeepOnet surrogate for 1D Advection Equation .

Equation: 
    U_t + v U_x = 0

"""

#%%
#Training Configuration - used as the config file for simvue.
configuration = {"Case": 'Advection',
                 "Field": 'u',
                 "Model": 'DeepOnet',
                 "Epochs": 10,
                 "Batch Size": 50,
                 "Optimizer": 'Adam',
                 "Learning Rate": 0.001,
                 "Scheduler Step": 1000,
                 "Scheduler Gamma": 0.5,
                 "Activation": 'Tanh',
                 "Normalisation Strategy": 'Identity',
                 "Layers": 4,
                 "Width": 256, 
                 "Variables":1, 
                 "Noise":0.0, 
                 "Loss Function": 'MSE',
                 }

import os
from simvue import Run
run = Run(mode='online')
run.init(folder="/Neural_PDE", tags=['NPDE', 'DeepONet', 'Tests'], metadata=configuration)

# Saving the current run file and the git hash of the repo
run.save_file(os.path.abspath(__file__), 'code')
import git
repo = git.Repo(search_parent_directories=True)
sha = repo.head.object.hexsha
run.update_metadata({'Git Hash': sha})

#Importing the necessary packages
import sys
import numpy as np
from tqdm import tqdm 
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib
import matplotlib.pyplot as plt
import time 
from timeit import default_timer
from tqdm import tqdm 

#Adding the NPDE package to the system python path
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

[simvue] Run speedy-station created
[simvue] Monitor in the UI at https://test.simvue.io/dashboard/runs/run/9hnF7YpzUBb5SCSF6FGHMS


In [2]:
#Importing the models and utilities. 
from Neural_PDE.Models.DeepOnet import *
from Neural_PDE.Utils.processing_utils import * 
from Neural_PDE.Utils.training_utils import * 

In [3]:
#Setting up locations. 
file_loc = os.getcwd()
data_loc = os.path.dirname(os.getcwd()) + '/Data'
model_loc = file_loc + '/Weights'
plot_loc = file_loc + '/Plots'
#Setting up the seeds and devices
torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_default_dtype(torch.float32)

In [4]:
#Generating the Datasets by running the simulation
t1 = default_timer()
from Neural_PDE.Numerical_Solvers.Advection.Advection_1D import *
from pyDOE import lhs

#Obtaining the exact and FD solution of the 1D Advection Equation. 

#Obtaining the exact and FD solution of the 1D Advection Equation. 
Nx = 200 #Number of x-points
Nt = 50 #Number of time instances 
x_min, x_max = 0.0, 2.0 #X min and max
t_end = 0.5 #time length
v = 1.0
sim = Advection_1d(Nx, Nt, x_min, x_max, t_end) 
dt, dx = sim.dt, sim.dx

n_sims = 10

lb = np.asarray([0.5, 50]) #pos, amplitude
ub = np.asarray([1.0, 200])

params = lb + (ub - lb) * lhs(2, n_sims)

u_sol = []
for ii in tqdm(range(n_sims)):
    xc = params[ii, 0]
    amp = params[ii, 1]
    x, t, u_soln, u_exact = sim.solve(xc, amp, v)
    u_sol.append(u_soln)

u_sol = np.asarray(u_sol)
u_sol = u_sol[:, :, 1:-2]
x = x[1:-2]

100%|██████████| 10/10 [00:00<00:00, 85.89it/s]


In [5]:
#Setting up the Data for DeepOnet
#Class that takes in the simulation solutions, X Mesh and the initial (sensor) locations and gives you a dataset
class DON_Dataset(Dataset):
    def __init__(self, u, x, t,  initial_locations):
        self.u = u
        self.x = x
        self.t = t
        self.X, self.T = torch.meshgrid(x,t, indexing='ij')
        self.x_loc = torch.column_stack((self.X.flatten(), self.T.flatten()))
        self.initial_locations = initial_locations

    def __len__(self):
        return self.u.shape[0]

    def __getitem__(self, idx):
        u_sample = self.u[idx]
        u_ic = u_sample[0].flatten()
        u_init= u_ic[self.initial_locations]
        
        return torch.FloatTensor(u_init), torch.FloatTensor(u_sample)

In [6]:
x = torch.tensor(x, dtype=torch.float32)
t = torch.linspace(0, t_end, 50)
X,T = torch.meshgrid(x,t)
X_loc = torch.column_stack((X.flatten(), T.flatten()))
initial_locations = np.arange(0, len(x))
u_sol = torch.tensor(u_sol, dtype=torch.float32)

/home/ir-gopa2/miniconda3/envs/torch_env/lib/python3.10/site-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /opt/conda/conda-bld/pytorch_1711403388920/work/aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [7]:
#Normalising the train and test datasets with the preferred normalisation. 

norm_strategy = configuration['Normalisation Strategy']

if norm_strategy == 'Min-Max':
    normalizer = MinMax_Normalizer
elif norm_strategy == 'Range':
    normalizer = RangeNormalizer
elif norm_strategy == 'Gaussian':
    normalizer = GaussianNormalizer
elif norm_strategy == 'Identity':
    normalizer = Identity

normalizer = normalizer(u_sol)

# #Saving Normalisation 
# saved_normalisations = model_loc + '/' + configuration['Model'] + '_' + configuration['Case'] + '_' +run.name + '_' + 'norms.npz'

# np.savez(saved_normalisations, 
#         a= normalizer.a.numpy(), b = normalizer.b.numpy()
# )

# run.save_file(saved_normalisations, 'output')

In [8]:
#Setting up the Datasets nad Loaders. 
dataset = DON_Dataset(normalizer.encode(u_sol), x, t, initial_locations)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=configuration['Batch Size'], shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=configuration['Batch Size'])

t2 = default_timer()
print('preprocessing finished, time used:', t2-t1)

preprocessing finished, time used: 0.43273214693181217


In [9]:
################################################################
# training and evaluation
################################################################
model = DeepONet(in_branch=len(x),
        width_branch=configuration['Width'],
        layers_branch=configuration['Layers'], 
        out_branch=configuration['Width'],
        in_trunk=2,
        width_trunk=configuration['Width'],
        layers_trunk=configuration['Layers'], 
        out_trunk=configuration['Width'])

model.to(device)

# run.update_metadata({'Number of Params': int(model.count_params())})
# print("Number of model params : " + str(model.count_params()))

#Setting up the optimizer and scheduler, loss and epochs 
optimizer = torch.optim.Adam(model.parameters(), lr=configuration['Learning Rate'], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=configuration['Scheduler Step'], gamma=configuration['Scheduler Gamma'])
loss_func = torch.nn.MSELoss()
epochs = configuration['Epochs']

In [10]:
####################################
#Training Loop 
####################################

def train_one_epoch_don(model, train_loader, test_loader, loss_func, optimizer):
    model.train()
    train_loss = 0
    for u0, yy in train_loader:
        optimizer.zero_grad()
        br = u0.unsqueeze(1).to(device)
        for tt in range(len(t)):
            tr = torch.FloatTensor(torch.column_stack((x.flatten(), torch.ones(x.shape)*tt))).to(device)
            tr = tr.unsqueeze(0).repeat(br.shape[0], 1, 1)
            y = yy[:,tt].to(device)
            im = model(br, tr)
            # print(br.shape, tr.shape, y.shape, im.shape)
            loss = loss_func(im, y)
 
        loss.backward()
        # torch.nn.utils.clip_grad_norm(parameters=model.parameters(), max_norm=max_grad_clip_norm, norm_type=2.0)
        optimizer.step()

        train_loss += loss.item()
    
    # Validation Loop
    test_loss = 0
    with torch.no_grad():
        for u0, yy in test_loader:
            br = u0.unsqueeze(1).to(device)
            for tt in range(len(t)):
                tr = torch.FloatTensor(torch.column_stack((x.flatten(), torch.ones(x.shape)*tt))).to(device)
                tr = tr.unsqueeze(0).repeat(br.shape[0], 1, 1)
                y = yy[:,tt].to(device)
                im = model(br, tr)
                test_loss += loss_func(im, y)
                
    return train_loss, test_loss #remember to divide the ntrain/ntest and num_vars at the other end before logging.


def validation_don(model, u0, yy):
    with torch.no_grad():
        br = u0.to(device)
        pred = np.zeroes(yy.shape)
        for tt in range(len(t)):
            x_loc = torch.FloatTensor(torch.column_stack((x.flatten(), torch.ones(x.shape)*tt))).to(device)
            y = yy[:,tt].to(device)
            im = model(br, x_loc)
            pred[tt] = im

        pred = torch.FloatTensor(pred)
            
        # Performance Metrics
        MSE_error = (yy - pred).pow(2).mean()
        MAE_error = torch.abs(yy - pred).mean()

    return pred, MSE_error, MAE_error


start_time = default_timer()
for ep in range(epochs): #Training Loop - Epochwise

    model.train()
    t1 = default_timer()
    train_loss, test_loss = train_one_epoch_don(model, train_loader, test_loader, loss_func, optimizer)
    t2 = default_timer()

    train_loss = train_loss # / ntrain / num_vars
    test_loss = test_loss #/ ntest / num_vars

    print(f"Epoch {ep}, Time Taken: {round(t2-t1,3)}, Train Loss: {round(train_loss, 5)}, Test Loss: {round(test_loss.item(),5)}")
    run.log_metrics({'Train Loss': train_loss, 'Test Loss': test_loss})
    
    scheduler.step()

train_time = default_timer() - start_time

Epoch 0, Time Taken: 0.353, Train Loss: 0.0857, Test Loss: 205.76823
Epoch 1, Time Taken: 0.346, Train Loss: 7.00023, Test Loss: 11.87477
Epoch 2, Time Taken: 0.343, Train Loss: 0.23297, Test Loss: 5.12969
Epoch 3, Time Taken: 0.356, Train Loss: 0.22648, Test Loss: 6.4857
Epoch 4, Time Taken: 0.352, Train Loss: 0.18529, Test Loss: 2.68514
Epoch 5, Time Taken: 0.348, Train Loss: 0.05772, Test Loss: 4.22177
Epoch 6, Time Taken: 0.348, Train Loss: 0.20041, Test Loss: 4.16481
Epoch 7, Time Taken: 0.345, Train Loss: 0.15565, Test Loss: 2.55669
Epoch 8, Time Taken: 0.344, Train Loss: 0.05303, Test Loss: 3.3148
Epoch 9, Time Taken: 0.346, Train Loss: 0.13208, Test Loss: 3.25811


In [11]:
            tr = tr.unsqueeze(0).repeat(br.shape[0], 1, 1)


NameError: name 'tr' is not defined

In [12]:
test_u0, test_u_encoded = next(iter(test_loader))


In [13]:
test_u0.shape

torch.Size([2, 200])